# 🎯 Entraînement Long Terme pour Diarisation de Locuteurs

Ce notebook explore les améliorations pour capturer les dépendances long terme en diarisation de locuteurs :

## 🎓 Approches testées :
1. **Architecture Long Range TCN** - TCN optimisé avec champ réceptif étendu
2. **Curriculum Learning** - Entraînement progressif avec segments de plus en plus longs  
3. **Attention Temporelle** - Mécanisme d'attention pour les très longues séquences
4. **Régularisation Adaptative** - Ajustement des hyperparamètres selon la longueur

## 🔍 Problème identifié :
- **Overfitting** sur segments courts (4s)
- **Champ réceptif limité** (~2s avec TCN actuel)
- **Pas de compréhension long terme** (10-30s nécessaires)

## 1. 🖥️ Configuration de l'environnement CUDA

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import OneCycleLR
from torch.cuda.amp import GradScaler, autocast
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json
import time
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# Configuration du style des graphiques
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("🚀 Configuration de l'environnement...")
print(f"📍 Répertoire de travail: {os.getcwd()}")
print(f"🐍 Version Python: {sys.version}")
print(f"🔥 Version PyTorch: {torch.__version__}")
print(f"🖥️ Nombre de CPU: {os.cpu_count()}")

# Vérification CUDA
if torch.cuda.is_available():
    print(f"✅ CUDA disponible!")
    print(f"🎮 GPU: {torch.cuda.get_device_name()}")
    print(f"💾 Mémoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"🔋 CUDA Capability: {torch.cuda.get_device_capability()}")
    device = torch.device('cuda')
    
    # Optimisations CUDA
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    
else:
    print("⚠️ CUDA non disponible, utilisation du CPU")
    device = torch.device('cpu')

print(f"🎯 Device sélectionné: {device}")

In [ ]:
# Ajout du répertoire src au path Python
sys.path.append('/home/dev/Speaker-diarization-/src')

# Import des modules locaux
try:
    from simple_tcn_model import SimpleDiarizationTCN
    from long_range_tcn import LongRangeDiarizationModel
    from voxconverse_dataset import create_voxconverse_dataloaders
    from simple_losses import create_loss_function
    from simple_metrics import create_metrics
    from progressive_training import create_progressive_training_config
    print("✅ Tous les modules importés avec succès!")
except ImportError as e:
    print(f"❌ Erreur d'import: {e}")
    print("Vérifiez que les fichiers sont présents dans le dossier src/")

## 2. 📊 Analyse du modèle actuel et de l'overfitting

In [ ]:
# Analyse des résultats précédents
def load_training_report():
    """Charge le rapport d'entraînement précédent."""
    try:
        with open('/home/dev/Speaker-diarization-/optimized_checkpoints_aggressive/training_report.json', 'r') as f:
            report = json.load(f)
        return report
    except FileNotFoundError:
        print("❌ Rapport d'entraînement non trouvé")
        return None

def analyze_current_model():
    """Analyse les limitations du modèle actuel."""
    print("🔍 ANALYSE DU MODÈLE ACTUEL")
    print("=" * 40)
    
    # Modèle actuel
    current_model = SimpleDiarizationTCN(
        input_dim=80,
        hidden_channels=[128, 256, 256, 512, 512],
        kernel_size=3,
        num_speakers=4,
        dropout=0.1
    )
    
    # Calcul du champ réceptif
    dilations = [2**i for i in range(5)]  # [1, 2, 4, 8, 16]
    kernel_size = 3
    theoretical_rf = sum((kernel_size - 1) * d for d in dilations)
    rf_seconds = theoretical_rf * 0.02  # 20ms par frame
    
    print(f"📏 Champ réceptif actuel:")
    print(f"   Dilatations: {dilations}")
    print(f"   Frames: {theoretical_rf}")
    print(f"   Durée: {rf_seconds:.1f}s")
    print(f"   Paramètres: {current_model.get_num_params():,}")
    
    # Charge les résultats précédents
    report = load_training_report()
    if report:
        print(f"\n📈 Résultats précédents:")
        print(f"   Meilleur val loss: {report['performance']['best_val_loss']:.6f}")
        print(f"   Temps d'entraînement: {report['performance']['total_training_time']:.1f}s")
        print(f"   Overfitting probable: Loss très faible")
    
    return {
        'model': current_model,
        'receptive_field_seconds': rf_seconds,
        'num_params': current_model.get_num_params(),
        'report': report
    }

# Analyse
current_analysis = analyze_current_model()

## 3. 🏗️ Comparaison des architectures

In [ ]:
def compare_architectures():
    """Compare les différentes architectures."""
    print("🏗️ COMPARAISON DES ARCHITECTURES")
    print("=" * 50)
    
    # Modèle actuel
    current_model = SimpleDiarizationTCN(
        input_dim=80,
        hidden_channels=[128, 256, 256, 512, 512],
        kernel_size=3,
        num_speakers=4,
        dropout=0.1
    )
    
    # Modèle long terme
    long_range_model = LongRangeDiarizationModel(
        input_dim=80,
        num_speakers=4,
        max_sequence_length=30.0
    )
    
    models = {
        'Simple TCN (Actuel)': current_model,
        'Long Range TCN': long_range_model
    }
    
    comparison_data = []
    
    print(f"{'Modèle':<20} {'Paramètres':<12} {'Champ réceptif':<15} {'Mémoire (MB)':<12}")
    print("-" * 65)
    
    for name, model in models.items():
        num_params = sum(p.numel() for p in model.parameters())
        
        # Calcul de la mémoire
        param_size = sum(p.numel() * p.element_size() for p in model.parameters())
        memory_mb = param_size / (1024 * 1024)
        
        # Champ réceptif
        if hasattr(model, 'get_receptive_field_info'):
            rf_info = model.get_receptive_field_info()
            rf_str = f"{rf_info['receptive_field_seconds']:.1f}s"
            rf_seconds = rf_info['receptive_field_seconds']
        else:
            rf_str = "~2.0s"
            rf_seconds = 2.0
        
        print(f"{name:<20} {num_params:>8,} {rf_str:>12} {memory_mb:>8.1f}")
        
        comparison_data.append({
            'model': name,
            'params': num_params,
            'receptive_field': rf_seconds,
            'memory_mb': memory_mb
        })
    
    return comparison_data, models

# Comparaison
comparison_data, models = compare_architectures()

In [ ]:
# Test de performance des modèles
def benchmark_models(models, test_lengths=[200, 400, 800, 1500]):
    """Benchmark les modèles sur différentes longueurs."""
    print("\n⚡ BENCHMARK DE PERFORMANCE")
    print("=" * 40)
    
    batch_size = 4
    input_dim = 80
    
    results = {}
    
    for model_name, model in models.items():
        print(f"\n🧪 Test {model_name}:")
        model.eval()
        model.to(device)
        
        model_results = []
        
        for length in test_lengths:
            duration = length * 0.02
            x = torch.randn(batch_size, input_dim, length).to(device)
            
            # Warmup
            with torch.no_grad():
                _ = model(x)
            
            # Mesure du temps
            torch.cuda.synchronize() if device.type == 'cuda' else None
            start_time = time.time()
            
            with torch.no_grad():
                vad_out, osd_out = model(x)
            
            torch.cuda.synchronize() if device.type == 'cuda' else None
            inference_time = time.time() - start_time
            
            # Mesure de la mémoire GPU
            if device.type == 'cuda':
                memory_used = torch.cuda.max_memory_allocated() / 1e6  # MB
                torch.cuda.reset_peak_memory_stats()
            else:
                memory_used = 0
            
            model_results.append({
                'duration': duration,
                'length': length,
                'inference_time': inference_time * 1000,  # ms
                'memory_mb': memory_used,
                'vad_shape': vad_out.shape,
                'osd_shape': osd_out.shape
            })
            
            print(f"   {duration:4.1f}s: {inference_time*1000:6.1f}ms, {memory_used:6.1f}MB")
        
        results[model_name] = model_results
    
    return results

# Benchmark
benchmark_results = benchmark_models(models)

## 4. 🎓 Configuration du Curriculum Learning

In [ ]:
# Configuration du curriculum learning
def create_curriculum_config():
    """Crée la configuration pour l'entraînement curriculum."""
    config = {
        'model': {
            'input_dim': 80,
            'hidden_channels': [128, 256, 256, 512, 512, 512],  # Architecture étendue
            'kernel_size': 3,
            'num_speakers': 8,  # CORRECTION: 8 speakers pour correspondre au dataset VoxConverse
            'dropout': 0.1,
            'use_long_range': True
        },
        'curriculum': {
            'schedule': [
                    
                (0, 10.0, 0.25), 
                (1, 30.0, 0.2), 
                (2, 60.0, 0.1),  

            ]
        },
        'training': {
            'total_epochs': 25,
            'base_batch_size': 16,
            'base_lr': 0.001,
            'weight_decay': 0.0001,
            'gradient_clip': 1.0
        },
        'optimization': {
            'use_amp': True,
            'use_compile': False,
            'memory_efficient': True
        }
    }
    return config

def visualize_curriculum_schedule(config):
    """Visualise le planning du curriculum."""
    schedule = config['curriculum']['schedule']
    total_epochs = config['training']['total_epochs']
    
    # Calcule l'évolution des paramètres
    epochs = list(range(total_epochs))
    segment_durations = []
    hop_ratios = []
    batch_sizes = []
    learning_rates = []
    
    for epoch in epochs:
        # Trouve la configuration actuelle
        current_duration = schedule[0][1]
        current_hop = schedule[0][2]
        
        for epoch_start, duration, hop_ratio in schedule:
            if epoch >= epoch_start:
                current_duration = duration
                current_hop = hop_ratio
        
        segment_durations.append(current_duration)
        hop_ratios.append(current_hop)
        
        # Batch size adaptatif (plus petit pour segments longs)
        if current_duration <= 4.0:
            batch_size = 16
        elif current_duration <= 8.0:
            batch_size = 8
        elif current_duration <= 16.0:
            batch_size = 4
        else:
            batch_size = 2
        batch_sizes.append(batch_size)
        
        # Learning rate adaptatif
        base_lr = 0.001
        stage = len([s for s in schedule if epoch >= s[0]]) - 1
        lr = base_lr * (0.9 ** stage)  # Décroissance par stage
        learning_rates.append(lr)
    
    # Graphiques
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Durée des segments
    axes[0, 0].plot(epochs, segment_durations, 'o-', linewidth=2, markersize=4)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Durée des segments (s)')
    axes[0, 0].set_title('📏 Évolution de la durée des segments')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Hop ratio
    axes[0, 1].plot(epochs, [h*100 for h in hop_ratios], 'o-', linewidth=2, markersize=4, color='orange')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Hop ratio (%)')
    axes[0, 1].set_title('🔄 Évolution du hop ratio')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Batch size
    axes[1, 0].plot(epochs, batch_sizes, 'o-', linewidth=2, markersize=4, color='green')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Batch Size')
    axes[1, 0].set_title('📦 Batch size adaptatif')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Learning rate
    axes[1, 1].plot(epochs, learning_rates, 'o-', linewidth=2, markersize=4, color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_title('📈 Learning rate adaptatif')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Affichage du planning
    print("🎓 PLANNING DU CURRICULUM LEARNING")
    print("=" * 50)
    for i, (epoch_start, duration, hop_ratio) in enumerate(schedule):
        print(f"Stage {i}: Epoch {epoch_start:2d}+ → {duration:4.1f}s segments (hop {hop_ratio*100:2.0f}%)")
    
    return {
        'epochs': epochs,
        'segment_durations': segment_durations,
        'hop_ratios': hop_ratios,
        'batch_sizes': batch_sizes,
        'learning_rates': learning_rates
    }

# Configuration et visualisation
curriculum_config = create_curriculum_config()
print(f"🔧 Configuration curriculum créée:")
print(f"   Nombre de speakers: {curriculum_config['model']['num_speakers']}")
print(f"   Stages curriculum: {len(curriculum_config['curriculum']['schedule'])}")

curriculum_viz = visualize_curriculum_schedule(curriculum_config)

## 5. 📥 Préparation des données avec curriculum

In [ ]:
def create_adaptive_dataloader(segment_duration, hop_ratio, batch_size, split='dev'):
    """Crée un dataloader adapté aux paramètres du curriculum."""
    
    print(f"📊 Création dataloader: {segment_duration}s segments, hop {hop_ratio*100:.0f}%, batch {batch_size}")
    
    try:
        # Configuration du dataloader
        dataloader_config = {
            'batch_size': batch_size,
            'num_workers': 16,  
            'segment_duration': segment_duration,
            'hop_duration': segment_duration * hop_ratio,
            'pin_memory': True if device.type == 'cuda' else False,
            'persistent_workers': True,  
            'validation_split': 0.1
        }
        
        # Crée les dataloaders
        if split == 'train':
            train_loader, _ = create_voxconverse_dataloaders(**dataloader_config)
            return train_loader
        else:
            _, val_loader = create_voxconverse_dataloaders(**dataloader_config)
            return val_loader
            
    except Exception as e:
        print(f"❌ Erreur création dataloader: {e}")
        print("💡 Utilisation d'un dataloader de test avec données synthétiques")
        


## 6. 🚀 Entraînement avec Curriculum Learning

In [ ]:
class CurriculumTrainer:
    """Entraîneur avec curriculum learning intégré."""
    
    def __init__(self, config, device):
        self.config = config
        self.device = device
        
        # Modèle avec le bon nombre de speakers
        self.model = LongRangeDiarizationModel(
            input_dim=config['model']['input_dim'],
            num_speakers=config['model']['num_speakers'],  # 8 speakers
            max_sequence_length=30.0
        ).to(device)
        
        print(f"🏗️ Modèle créé: {sum(p.numel() for p in self.model.parameters()):,} paramètres")
        print(f"🎯 Nombre de speakers: {config['model']['num_speakers']}")
        
        # Optimiseur et loss
        self.optimizer = optim.AdamW(
            self.model.parameters(),
            lr=config['training']['base_lr'],
            weight_decay=config['training']['weight_decay']
        )
        
        self.criterion = create_loss_function({
            'type': 'simple',
            'vad_weight': 1.0,
            'osd_weight': 1.2,
            'focal_gamma': 2.0,
            'focal_alpha': 0.25,
            'label_smoothing': 0.05
        })
        
        # Mixed precision
        self.use_amp = config['optimization']['use_amp'] and device.type == 'cuda'
        self.scaler = GradScaler() if self.use_amp else None
        
        # Tracking
        self.history = []
        self.current_stage = 0
        
    def get_current_curriculum_config(self, epoch):
        """Obtient la configuration curriculum pour l'epoch actuel."""
        schedule = self.config['curriculum']['schedule']
        
        for i, (start_epoch, duration, hop_ratio) in enumerate(schedule):
            if epoch >= start_epoch:
                current_config = {
                    'stage': i,
                    'epoch_start': start_epoch,
                    'segment_duration': duration,
                    'hop_ratio': hop_ratio
                }
            else:
                break
        
        return current_config
    
    def adjust_learning_rate(self, epoch, stage_info):
        """Ajuste le learning rate selon l'étape du curriculum."""
        base_lr = self.config['training']['base_lr']
        
        # Réduit le LR pour les segments plus longs
        stage_multiplier = 1.0 / (1.0 + 0.1 * stage_info['stage'])
        
        # Warmup pour chaque nouvelle étape
        epochs_in_stage = epoch - stage_info['epoch_start']
        if epochs_in_stage < 2:  # 2 epochs de warmup
            lr_multiplier = 0.5 + 0.5 * (epochs_in_stage / 2)
        else:
            lr_multiplier = 1.0
        
        new_lr = base_lr * stage_multiplier * lr_multiplier
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = new_lr
        
        return new_lr
    
    def train_epoch(self, epoch):
        """Entraîne une époque avec curriculum."""
        # Configuration curriculum
        stage_info = self.get_current_curriculum_config(epoch)
        current_lr = self.adjust_learning_rate(epoch, stage_info)
        
        # Batch size adaptatif
        base_batch_size = self.config['training']['base_batch_size']
        if stage_info['segment_duration'] <= 4.0:
            batch_size = base_batch_size
        elif stage_info['segment_duration'] <= 8.0:
            batch_size = max(4, base_batch_size // 2)
        else:
            batch_size = max(2, base_batch_size // 4)
        
        print(f"\n🎓 Epoch {epoch} - Stage {stage_info['stage']}")
        print(f"   Segments: {stage_info['segment_duration']:.1f}s")
        print(f"   Hop ratio: {stage_info['hop_ratio']:.1%}")
        print(f"   Batch size: {batch_size}")
        print(f"   Learning rate: {current_lr:.6f}")
        
        # Crée le dataloader pour cette configuration
        train_loader = create_adaptive_dataloader(
            stage_info['segment_duration'],
            stage_info['hop_ratio'],
            batch_size,
            split='train'
        )
        
        # Entraînement
        self.model.train()
        total_loss = 0
        num_batches = len(train_loader)
        
        pbar = tqdm(train_loader, desc=f'Stage {stage_info["stage"]} Training')
        
        for batch_idx, batch in enumerate(pbar):
            features = batch['features'].to(self.device)
            vad_labels = batch['vad_labels'].to(self.device)
            osd_labels = batch['osd_labels'].to(self.device)
            
            # Debug des dimensions
            if batch_idx == 0:
                print(f"   📐 Features: {features.shape}")
                print(f"   📐 VAD labels: {vad_labels.shape}")
                print(f"   📐 OSD labels: {osd_labels.shape}")
            
            self.optimizer.zero_grad()
            
            # Forward pass
            if self.use_amp:
                with autocast():
                    vad_pred, osd_pred = self.model(features)
                    if batch_idx == 0:
                        print(f"   📐 VAD pred: {vad_pred.shape}")
                        print(f"   📐 OSD pred: {osd_pred.shape}")
                    loss_dict = self.criterion(vad_pred, osd_pred, vad_labels, osd_labels)
                    loss = loss_dict['total_loss']
                
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                vad_pred, osd_pred = self.model(features)
                if batch_idx == 0:
                    print(f"   📐 VAD pred: {vad_pred.shape}")
                    print(f"   📐 OSD pred: {osd_pred.shape}")
                loss_dict = self.criterion(vad_pred, osd_pred, vad_labels, osd_labels)
                loss = loss_dict['total_loss']
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.optimizer.step()
            
            total_loss += loss.item()
            
            pbar.set_postfix({
                'Loss': f"{loss.item():.4f}",
                'VAD': f"{loss_dict['vad_loss'].item():.4f}",
                'OSD': f"{loss_dict['osd_loss'].item():.4f}",
                'LR': f"{current_lr:.6f}"
            })
        
        avg_loss = total_loss / num_batches
        
        # Sauvegarde l'historique
        self.history.append({
            'epoch': epoch,
            'stage': stage_info['stage'],
            'segment_duration': stage_info['segment_duration'],
            'hop_ratio': stage_info['hop_ratio'],
            'batch_size': batch_size,
            'learning_rate': current_lr,
            'train_loss': avg_loss
        })
        
        return avg_loss
    
    def train_curriculum(self, num_epochs):
        """Entraînement complet avec curriculum."""
        print(f"🚀 Début entraînement curriculum ({num_epochs} epochs)")
        
        for epoch in range(num_epochs):
            train_loss = self.train_epoch(epoch)
            print(f"Epoch {epoch}: Loss = {train_loss:.6f}")
        
        print("✅ Entraînement terminé!")
        return self.history

# Création du trainer avec la nouvelle configuration
trainer = CurriculumTrainer(curriculum_config, device)
print("🎯 Trainer curriculum créé avec succès!")
print(f"🔧 Configuration: {curriculum_config['model']['num_speakers']} speakers")

In [ ]:
# 🧪 ENTRAÎNEMENT DE TEST (quelques epochs pour validation)
print("🧪 TEST D'ENTRAÎNEMENT CURRICULUM")
print("=" * 40)

# Entraînement court pour tester le pipeline
test_epochs = 10
history = trainer.train_curriculum(test_epochs)

## 7. 📊 Monitoring et visualisation des résultats

In [ ]:
def visualize_training_progress(history):
    """Visualise les progrès de l'entraînement curriculum."""
    if not history:
        print("❌ Pas d'historique d'entraînement disponible")
        return
    
    epochs = [h['epoch'] for h in history]
    stages = [h['stage'] for h in history]
    durations = [h['segment_duration'] for h in history]
    losses = [h['train_loss'] for h in history]
    lrs = [h['learning_rate'] for h in history]
    batch_sizes = [h['batch_size'] for h in history]
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Loss d'entraînement
    axes[0, 0].plot(epochs, losses, 'o-', linewidth=2, markersize=6)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Training Loss')
    axes[0, 0].set_title('📉 Évolution de la loss')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Durée des segments
    axes[0, 1].plot(epochs, durations, 'o-', linewidth=2, markersize=6, color='orange')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Segment Duration (s)')
    axes[0, 1].set_title('📏 Progression des segments')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Learning rate
    axes[0, 2].plot(epochs, lrs, 'o-', linewidth=2, markersize=6, color='red')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('Learning Rate')
    axes[0, 2].set_title('📈 Learning Rate Schedule')
    axes[0, 2].set_yscale('log')
    axes[0, 2].grid(True, alpha=0.3)
    
    # Stages du curriculum
    axes[1, 0].plot(epochs, stages, 'o-', linewidth=2, markersize=6, color='green')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Curriculum Stage')
    axes[1, 0].set_title('🎓 Stages du curriculum')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Batch sizes
    axes[1, 1].plot(epochs, batch_sizes, 'o-', linewidth=2, markersize=6, color='purple')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Batch Size')
    axes[1, 1].set_title('📦 Batch Size Adaptatif')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Loss par stage (scatter plot)
    colors = plt.cm.viridis(np.array(stages) / max(stages) if stages else [0])
    scatter = axes[1, 2].scatter(durations, losses, c=colors, s=100, alpha=0.7, edgecolors='black')
    axes[1, 2].set_xlabel('Segment Duration (s)')
    axes[1, 2].set_ylabel('Training Loss')
    axes[1, 2].set_title('🎯 Loss vs Durée des segments')
    axes[1, 2].grid(True, alpha=0.3)
    
    # Colorbar pour le scatter plot
    cbar = plt.colorbar(scatter, ax=axes[1, 2])
    cbar.set_label('Curriculum Stage')
    
    plt.tight_layout()
    plt.show()
    
    # Statistiques
    print("\n📊 STATISTIQUES D'ENTRAÎNEMENT")
    print("=" * 40)
    print(f"Epochs totaux: {len(history)}")
    print(f"Stages parcourus: {len(set(stages))}")
    print(f"Loss initiale: {losses[0]:.6f}")
    print(f"Loss finale: {losses[-1]:.6f}")
    print(f"Amélioration: {((losses[0] - losses[-1]) / losses[0] * 100):.1f}%")
    
    return fig

# Visualisation des résultats
if history:
    fig = visualize_training_progress(history)
else:
    print("⚠️ Pas d'historique à visualiser (entraînement pas encore lancé)")

In [ ]:
# 🔧 Monitoring GPU et mémoire
def monitor_gpu_usage():
    """Surveille l'utilisation GPU."""
    if device.type == 'cuda':
        print("\n🖥️ MONITORING GPU")
        print("=" * 30)
        
        # Mémoire GPU
        memory_allocated = torch.cuda.memory_allocated() / 1e9
        memory_reserved = torch.cuda.memory_reserved() / 1e9
        max_memory = torch.cuda.max_memory_allocated() / 1e9
        total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        
        print(f"💾 Mémoire allouée: {memory_allocated:.2f} GB")
        print(f"💾 Mémoire réservée: {memory_reserved:.2f} GB")
        print(f"💾 Pic mémoire: {max_memory:.2f} GB")
        print(f"💾 Mémoire totale: {total_memory:.2f} GB")
        print(f"📊 Utilisation: {(memory_allocated/total_memory)*100:.1f}%")
        
        return {
            'allocated_gb': memory_allocated,
            'reserved_gb': memory_reserved,
            'max_allocated_gb': max_memory,
            'total_gb': total_memory,
            'utilization': memory_allocated/total_memory
        }
    else:
        print("⚠️ Monitoring GPU non disponible (CPU mode)")
        return None

# Monitoring actuel
gpu_stats = monitor_gpu_usage()

## 8. 💾 Sauvegarde et checkpoints

In [ ]:
def save_checkpoint(trainer, epoch, save_dir='long_range_checkpoints'):
    """Sauvegarde un checkpoint du modèle."""
    os.makedirs(save_dir, exist_ok=True)
    
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': trainer.model.state_dict(),
        'optimizer_state_dict': trainer.optimizer.state_dict(),
        'config': trainer.config,
        'history': trainer.history,
        'model_info': {
            'num_parameters': sum(p.numel() for p in trainer.model.parameters()),
            'architecture': 'LongRangeDiarizationModel',
            'curriculum_enabled': True
        }
    }
    
    # Sauvegarde
    checkpoint_path = os.path.join(save_dir, f'long_range_model_epoch_{epoch}.pth')
    torch.save(checkpoint, checkpoint_path)
    
    print(f"💾 Checkpoint sauvegardé: {checkpoint_path}")
    return checkpoint_path

def save_training_report(trainer, save_dir='long_range_checkpoints'):
    """Sauvegarde un rapport d'entraînement détaillé."""
    os.makedirs(save_dir, exist_ok=True)
    
    report = {
        'timestamp': datetime.now().isoformat(),
        'config': trainer.config,
        'model_info': {
            'architecture': 'LongRangeDiarizationModel',
            'num_parameters': sum(p.numel() for p in trainer.model.parameters()),
            'model_size_mb': sum(p.numel() * p.element_size() for p in trainer.model.parameters()) / (1024*1024)
        },
        'training_history': trainer.history,
        'performance_summary': {
            'total_epochs': len(trainer.history),
            'curriculum_stages': len(set(h['stage'] for h in trainer.history)) if trainer.history else 0,
            'final_loss': trainer.history[-1]['train_loss'] if trainer.history else None,
            'loss_improvement': (trainer.history[0]['train_loss'] - trainer.history[-1]['train_loss']) if len(trainer.history) > 1 else 0
        },
        'gpu_stats': monitor_gpu_usage() if device.type == 'cuda' else None
    }
    
    report_path = os.path.join(save_dir, 'long_range_training_report.json')
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    
    print(f"📄 Rapport sauvegardé: {report_path}")
    return report_path

# Sauvegarde des résultats actuels
print("💾 SAUVEGARDE DES RÉSULTATS")
print("=" * 30)

if hasattr(trainer, 'history') and trainer.history:
    # Sauvegarde checkpoint
    latest_epoch = trainer.history[-1]['epoch']
    checkpoint_path = save_checkpoint(trainer, latest_epoch)
    
    # Sauvegarde rapport
    report_path = save_training_report(trainer)
    
    print(f"✅ Sauvegarde terminée!")
    print(f"📁 Checkpoint: {checkpoint_path}")
    print(f"📁 Rapport: {report_path}")
else:
    print("⚠️ Pas d'historique d'entraînement à sauvegarder")